# 0.1.0 Corpus split for analysis & ML training

This notebook creates two subset corpora: one to retain for analysis, and one for machine learning training. The purpose of this step is to not introduce bias in the analysis stage by including examples from the training set. This way, the ML can train on a subset that is as similar to the analysis set as possible without introducing bias.


---

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def _find_workspace_root() -> Path:
    env_root = os.environ.get("WORKSPACE_ROOT", "").strip()
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "config" / "workspace.json").exists():
            return candidate
    for start in [Path.cwd(), *Path.cwd().parents]:
        if (start / "config" / "workspace.json").exists():
            return start
    raise FileNotFoundError(
        "Could not locate workspace root from WORKSPACE_ROOT or the current working directory."
    )


WORKSPACE_DIR = _find_workspace_root()
PAPER_ROOT = WORKSPACE_DIR.parent
RESEARCH_ROOT = PAPER_ROOT.parent
SHARED_ASSETS_DIR = RESEARCH_ROOT / "shared-assets"
SHARED_CODE_DIR = SHARED_ASSETS_DIR / "code"
if str(SHARED_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_CODE_DIR))

CODE_DIR = WORKSPACE_DIR / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from workspace_rooting.workspace_paths import canonical_workspace_paths

paths = canonical_workspace_paths(WORKSPACE_DIR)
WORKSPACE_DIR = paths["workspace"]
CODE_DIR = paths["code"]
CONFIG_DIR = paths["config"]
DATA_DIR = paths["data"]
OUTPUTS_DIR = paths["outputs"]
DOCS_DIR = paths["docs"]
LOCAL_DIR = paths["local"]
SHARED_ASSETS_DIR = paths["shared_assets"]
SHARED_CODE_DIR = SHARED_ASSETS_DIR / "code"

print("workspace_root:", WORKSPACE_DIR)
print("shared_assets_root:", SHARED_ASSETS_DIR)

import json
from pathlib import Path
import numpy as np
import pandas as pd
import hashlib
from datetime import datetime, timezone
import re

from collections import Counter

print("workspace_root:", WORKSPACE_DIR)

RUN_ID = "analysis_corpus"

MASTER_MANIFEST = LOCAL_DIR / "corpora" / "master_corpus" / "master_corpus_manifest.json"
MASTER_PARQUET = LOCAL_DIR / "corpora" / "master_corpus" / "master_corpus.parquet"

CORPUS_ROOT = LOCAL_DIR / "corpora" / RUN_ID
CORPUS_DIR = CORPUS_ROOT / "bins"
CORPUS_ROOT.mkdir(parents=True, exist_ok=True)
CORPUS_DIR.mkdir(parents=True, exist_ok=True)


def workspace_relative(path: Path) -> str:
    return str(path.resolve().relative_to(WORKSPACE_DIR)).replace("\\", "/")


print("MASTER_MANIFEST:", MASTER_MANIFEST,
      "MASTER_PARQUET:", MASTER_PARQUET,
      "CORPUS_DIR:", CORPUS_DIR,
      "CORPUS_ROOT:", CORPUS_ROOT)

ANALYSIS_START_YEAR = 1986
BIN_YEARS = 5
END_YEAR = 2025
EARLIEST_BACKFILL_YEAR = 1975

MIN_EXAMPLES_PER_BIN = 1300
MAX_EXAMPLES_PER_BIN = 1600
MAX_EXAMPLES_PER_BIBCODE = 2
SEED = 42
rng = np.random.default_rng(SEED)



 ## Functions, helpers and definitions 


---

In [ ]:
def bin_start_year(label: str) -> int:
    return int(str(label).split("-")[0])

def materialize_rows(df_subset: pd.DataFrame, time_bin_value: str) -> list[dict]:
    cols = ["sent", "start", "end", "bibcode", "has_math", "year", "mention_order"]
    rows = []
    for row in df_subset[cols].to_dict(orient="records"):
        rows.append({
            "sent": row["sent"],
            "start": int(row["start"]),
            "end": int(row["end"]),
            "bibcode": str(row["bibcode"]),
            "has_math": bool(row["has_math"]),
            "year": int(row["year"]),
            "time_bin": time_bin_value,
            "mention_order": int(row["mention_order"]),
        })
    return rows

def cap_per_bibcode(rows: list[dict], rng: np.random.Generator) -> list[dict]:
    by_bibcode = {}
    for row in rows:
        by_bibcode.setdefault(row["bibcode"], []).append(row)

    capped = []
    for bibcode, exs in by_bibcode.items():
        if len(exs) > MAX_EXAMPLES_PER_BIBCODE:
            keep_idx = rng.choice(len(exs), size=MAX_EXAMPLES_PER_BIBCODE, replace=False)
            capped.extend(exs[i] for i in keep_idx)
        else:
            capped.extend(exs)
    return capped

def year_balanced_sample(rows: list[dict], target_total: int, rng: np.random.Generator) -> list[dict]:
    if len(rows) <= target_total:
        return list(rows)

    by_year = {}
    for row in rows:
        by_year.setdefault(int(row["year"]), []).append(row)

    years = sorted(by_year)
    base_quota = max(1, target_total // len(years))
    selected = []
    leftovers = []

    for year in years:
        exs = by_year[year]
        if len(exs) <= base_quota:
            selected.extend(exs)
        else:
            keep_idx = rng.choice(len(exs), size=base_quota, replace=False)
            keep_idx = set(int(i) for i in keep_idx.tolist())
            selected.extend(exs[i] for i in keep_idx)
            leftovers.extend(exs[i] for i in range(len(exs)) if i not in keep_idx)

    remaining = target_total - len(selected)
    if remaining > 0 and leftovers:
        fill_idx = rng.choice(len(leftovers), size=min(remaining, len(leftovers)), replace=False)
        selected.extend(leftovers[i] for i in fill_idx)

    if len(selected) > target_total:
        keep_idx = rng.choice(len(selected), size=target_total, replace=False)
        selected = [selected[i] for i in keep_idx]

    return selected

def resolve_first_bin_from_master(master_df: pd.DataFrame, rng: np.random.Generator, min_examples: int, first_bin_end: int = 1990):
    rows = []
    for start_year in range(ANALYSIS_START_YEAR, EARLIEST_BACKFILL_YEAR - 1, -1):
        bin_label_out = f"{start_year}-{first_bin_end}"
        subset = master_df[
            master_df["year"].between(start_year, first_bin_end, inclusive="both")
        ].copy()
        subset = subset.sort_values(
            ["source_row_index", "mention_order", "bibcode", "start", "end"],
            kind="mergesort",
        )
        rows = materialize_rows(subset, time_bin_value=bin_label_out)
        rows = cap_per_bibcode(rows, rng)
        if len(rows) >= min_examples:
            return start_year, rows
    return None, rows

def write_jsonl(rows: list[dict], path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            row_out = dict(row)
            row_out.pop("mention_order", None)
            f.write(json.dumps(row_out, ensure_ascii=False))
            f.write("\n")

def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def load_main_corpus_index(corpus_dir: Path):
    by_bin_pairs = {}
    by_bin_bibcodes = {}
    by_bin_counts = {}

    for path in sorted(corpus_dir.glob("*.jsonl"), key=lambda p: bin_start_year(p.stem)):
        rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
        by_bin_counts[path.stem] = len(rows)

        pairs = set()
        bibcodes = set()
        for row in rows:
            sent = " ".join(row["sent"].split())
            bibcode = str(row.get("bibcode", ""))
            pairs.add((bibcode, sent))
            bibcodes.add(bibcode)

        by_bin_pairs[path.stem] = pairs
        by_bin_bibcodes[path.stem] = bibcodes

    return by_bin_pairs, by_bin_bibcodes, by_bin_counts

def resolve_main_overlap_label(bin_label_out: str, main_pairs_by_bin: dict[str, set]) -> str:
    if bin_label_out in main_pairs_by_bin:
        return bin_label_out

    try:
        bin_end = int(bin_label_out.split("-")[1])
    except Exception:
        return bin_label_out

    candidates = []
    for label in main_pairs_by_bin:
        try:
            if int(label.split("-")[1]) == bin_end:
                candidates.append(label)
        except Exception:
            continue

    if candidates:
        return min(candidates, key=bin_start_year)
    return bin_label_out

def select_shadow_examples(examples, bin_label_out, target_total, rng, main_pairs_by_bin, main_bibcodes_by_bin):
    main_label = resolve_main_overlap_label(bin_label_out, main_pairs_by_bin)
    main_pairs = main_pairs_by_bin.get(main_label, set())
    main_bibcodes = main_bibcodes_by_bin.get(main_label, set())

    fresh_pair_fresh_bib = []
    fresh_pair_overlap_bib = []
    overlap_pair = []

    for ex in examples:
        pair = (ex["bibcode"], ex["sent"])
        pair_overlap = pair in main_pairs
        bib_overlap = ex["bibcode"] in main_bibcodes

        if not pair_overlap and not bib_overlap:
            fresh_pair_fresh_bib.append(ex)
        elif not pair_overlap:
            fresh_pair_overlap_bib.append(ex)
        else:
            overlap_pair.append(ex)

    selected = []
    selected.extend(year_balanced_sample(fresh_pair_fresh_bib, min(target_total, len(fresh_pair_fresh_bib)), rng))
    remaining = target_total - len(selected)

    if remaining > 0:
        keep = year_balanced_sample(fresh_pair_overlap_bib, min(remaining, len(fresh_pair_overlap_bib)), rng)
        selected.extend(keep)
        remaining = target_total - len(selected)

    if remaining > 0:
        keep = year_balanced_sample(overlap_pair, min(remaining, len(overlap_pair)), rng)
        selected.extend(keep)

    selected_pairs = {(ex["bibcode"], ex["sent"]) for ex in selected}
    overlap_stats = {
        "available_total": len(examples),
        "selected_total": len(selected),
        "main_overlap_bin": main_label,
        "fresh_pair_fresh_bib_available": len(fresh_pair_fresh_bib),
        "fresh_pair_overlap_bib_available": len(fresh_pair_overlap_bib),
        "overlap_pair_available": len(overlap_pair),
        "fresh_pair_fresh_bib_selected": len([ex for ex in selected if ex["bibcode"] not in main_bibcodes and (ex["bibcode"], ex["sent"]) not in main_pairs]),
        "fresh_pair_overlap_bib_selected": len([ex for ex in selected if ex["bibcode"] in main_bibcodes and (ex["bibcode"], ex["sent"]) not in main_pairs]),
        "overlap_pair_selected": len(selected_pairs & main_pairs),
        "shadow_bibcode_overlap_rate": (
            sum(1 for ex in selected if ex["bibcode"] in main_bibcodes) / len(selected)
            if selected else 0.0
        ),
        "shadow_pair_overlap_rate": (
            len(selected_pairs & main_pairs) / len(selected)
            if selected else 0.0
        ),
    }
    return selected, overlap_stats

def highlight_span(sent: str, start: int, end: int) -> str:
    try:
        start = int(start)
        end = int(end)
        if 0 <= start < end <= len(sent):
            return sent[:start] + "[[DARK_MATTER]]" + sent[end:]
    except Exception:
        pass
    return sent

def classify_bucket(sent: str):
    text = (sent or "").lower()
    a = bool(A_PAT.search(text))
    b = bool(B_PAT.search(text))
    c = bool(C_PAT.search(text))
    n = sum([a, b, c])

    if n >= 2:
        return "mixed", {"A": a, "B": b, "C": c}
    if a:
        return "A", {"A": a, "B": b, "C": c}
    if b:
        return "B", {"A": a, "B": b, "C": c}
    if c:
        return "C", {"A": a, "B": b, "C": c}
    return "other", {"A": a, "B": b, "C": c}

def choose_without_replacement(candidates, n_take):
    if n_take <= 0 or not candidates:
        return []
    if len(candidates) <= n_take:
        return list(candidates)
    idx = SAMPLING_RNG.choice(len(candidates), size=n_take, replace=False)
    idx = sorted(int(i) for i in idx)
    return [candidates[i] for i in idx]

def load_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def example_key(row) -> str:
    sent = ' '.join(str(row['sent']).split())
    return f"{row['bibcode']}|||{sent}|||{int(row['start'])}|||{int(row['end'])}"

def materialize_lexical_rows(df_subset: pd.DataFrame, time_bin_value: str) -> list[dict]:
    cols = ["example_key", "sent", "start", "end", "bibcode", "has_math", "year"]
    rows = []
    for row in df_subset[cols].to_dict(orient="records"):
        rows.append({
            "example_key": str(row["example_key"]),
            "sent": row["sent"],
            "start": int(row["start"]),
            "end": int(row["end"]),
            "bibcode": str(row["bibcode"]),
            "has_math": bool(row["has_math"]),
            "year": int(row["year"]),
            "time_bin": time_bin_value,
        })
    return rows

def build_derived_corpus_manifest(
    *,
    run_id: str,
    output_dir: Path,
    master_manifest: dict,
    written_bins: list[str],
    final_bin_counts: dict[str, int],
    bin_content_hashes: dict[str, str],
    start_year: int,
    end_year: int,
    bin_years: int,
    filters: dict,
    sampling: dict,
    extra_fields: dict | None = None,
) -> dict:
    manifest = {
        "run_id": run_id,
        "source_parquet": master_manifest["source_parquet"],
        "source_master_run_id": master_manifest.get("master_run_id"),
        "source_master_parquet": workspace_relative(MASTER_PARQUET),
        "output_dir": workspace_relative(output_dir),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "start_year": start_year,
        "end_year": end_year,
        "bin_years": bin_years,
        "written_bins": written_bins,
        "n_written_bins": len(written_bins),
        "filters": filters,
        "sampling": sampling,
        "n_rows_after_filters": master_manifest.get("row_counts", {}).get("filtered_source_rows"),
        "final_bin_counts": final_bin_counts,
        "final_total_examples": int(sum(final_bin_counts.values())),
        "bin_content_hashes": bin_content_hashes,
    }
    if extra_fields:
        manifest.update(extra_fields)
    return manifest


# Analysis corpus


---

In [ ]:
master_df = pd.read_parquet(MASTER_PARQUET)

required_cols = {
    "source_row_index",
    "mention_order",
    "year",
    "time_bin_current",
    "bibcode",
    "sent",
    "start",
    "end",
    "has_math",
}
missing = required_cols - set(master_df.columns)
if missing:
    raise ValueError(f"Master mention table missing required columns: {sorted(missing)}")

old_bin_files = sorted(CORPUS_DIR.glob("*.jsonl"))
if old_bin_files:
    print(f"Removing {len(old_bin_files)} existing bin files from {CORPUS_DIR}")
    for p in old_bin_files:
        p.unlink()

written_bins = []
final_bin_counts = {}

fixed_bin_labels = sorted(
    master_df["time_bin_current"].dropna().astype(str).unique(),
    key=bin_start_year,
)

for bin_label in fixed_bin_labels:
    bin_end = int(bin_label.split("-")[1])

    if bin_label == f"{ANALYSIS_START_YEAR}-{ANALYSIS_START_YEAR + BIN_YEARS - 1}":
        resolved_start, rows = resolve_first_bin_from_master(
            master_df=master_df,
            rng=rng,
            min_examples=MIN_EXAMPLES_PER_BIN,
            first_bin_end=bin_end,
        )
        if resolved_start is None:
            print(f"{bin_label}: could not reach {MIN_EXAMPLES_PER_BIN}, skipping")
            continue
        bin_label_out = f"{resolved_start}-{bin_end}"
        rows = [{**row, "time_bin": bin_label_out} for row in rows]
    else:
        bin_label_out = bin_label
        subset = master_df[master_df["time_bin_current"] == bin_label].copy()
        subset = subset.sort_values(
            ["source_row_index", "mention_order", "bibcode", "start", "end"],
            kind="mergesort",
        )
        rows = materialize_rows(subset, time_bin_value=bin_label_out)
        rows = cap_per_bibcode(rows, rng)

        if len(rows) < MIN_EXAMPLES_PER_BIN:
            print(f"{bin_label}: only {len(rows)} rows after caps, skipping")
            continue

    if len(rows) > MAX_EXAMPLES_PER_BIN:
        rows = year_balanced_sample(rows, MAX_EXAMPLES_PER_BIN, rng)

    out_file = CORPUS_DIR / f"{bin_label_out}.jsonl"
    write_jsonl(rows, out_file)
    written_bins.append(bin_label_out)
    final_bin_counts[bin_label_out] = len(rows)

    yr_counts = pd.Series([r["year"] for r in rows]).value_counts().sort_index().to_dict()
    print(f"{bin_label_out}: wrote {len(rows)} rows -> {out_file} | per-year: {yr_counts}")

print("\nFinal bin counts:")
print(final_bin_counts)


## Build and save the derived manifest for analysis corpus

---

In [ ]:
MANIFEST_PATH = CORPUS_ROOT / "corpus_manifest.json"

with open(MASTER_MANIFEST, "r", encoding="utf-8") as f:
    master_manifest = json.load(f)

# Keep the derived manifest compatible with the production analysis_corpus schema.
prod_filter_keys = [
    "require_year",
    "require_abstract",
    "lowercase_abstract",
    "excluded_arxiv_categories",
    "excluded_doctypes",
    "biology_filter_terms",
]
derived_filters = {
    key: master_manifest["filters"][key]
    for key in prod_filter_keys
    if key in master_manifest.get("filters", {})
}

bin_content_hashes = {
    bin_label: file_sha256(CORPUS_DIR / f"{bin_label}.jsonl")
    for bin_label in written_bins
}

first_written_bin = written_bins[0] if written_bins else None
first_bin_start_actual = int(first_written_bin.split("-")[0]) if first_written_bin else None
first_bin_end_actual = int(first_written_bin.split("-")[1]) if first_written_bin else None

manifest = {
    "run_id": RUN_ID,
    "source_parquet": master_manifest["source_parquet"],
    "output_dir": workspace_relative(CORPUS_DIR),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),

    "start_year": ANALYSIS_START_YEAR,
    "end_year": END_YEAR,
    "bin_years": BIN_YEARS,
    "bin_labels_planned": [
        f"{ANALYSIS_START_YEAR + i * BIN_YEARS}-{min(ANALYSIS_START_YEAR + (i + 1) * BIN_YEARS - 1, END_YEAR)}"
        for i in range(((END_YEAR - ANALYSIS_START_YEAR) // BIN_YEARS) + 1)
    ],
    "written_bins": written_bins,
    "n_written_bins": len(written_bins),

    "first_bin_policy": {
        "fixed_end_year": ANALYSIS_START_YEAR + BIN_YEARS - 1,
        "analysis_start_year": ANALYSIS_START_YEAR,
        "earliest_backfill_year": EARLIEST_BACKFILL_YEAR,
        "first_bin_actual_start_year": first_bin_start_actual,
        "first_bin_actual_end_year": first_bin_end_actual,
        "first_bin_was_backfilled": (
            first_bin_start_actual is not None and first_bin_start_actual < ANALYSIS_START_YEAR
        ),
    },

    "filters": derived_filters,

    "sampling": {
        "min_examples_per_bin": MIN_EXAMPLES_PER_BIN,
        "max_examples_per_bin": MAX_EXAMPLES_PER_BIN,
        "max_examples_per_bibcode": MAX_EXAMPLES_PER_BIBCODE,
        "seed": SEED,
    },

    "n_rows_after_filters": master_manifest["row_counts"]["filtered_source_rows"],
    "final_bin_counts": final_bin_counts,
    "final_total_examples": int(sum(final_bin_counts.values())),
    "bin_content_hashes": bin_content_hashes,
}

with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(f"Saved corpus manifest -> {MANIFEST_PATH}")


## Analysis corpus created

The cells above built the main analysis corpus from the validated master mention table. The next section derives a separate corpus, supporting annotation, from the same master source so that LLM seeding and sciBERT classifier training can proceed without leaking back into the main inferential corpus.

---

---

# Shadow corpus

Below, we create a second corpus, non-overlapping with the corpus created above, stored internally as “shadow_corpus”. The purpose of *shadow corpus* is instrumental only. It's utility lies in methodological benefits rather than substantive interpretation.


## Derive shadow corpus from the validated master mention table

---

In [ ]:
MAIN_RUN_ID = "analysis_corpus"
MAIN_CORPUS_DIR = LOCAL_DIR / "corpora" / MAIN_RUN_ID / "bins"

SHADOW_RUN_ID = "shadow_corpus"
SHADOW_ROOT = LOCAL_DIR / "corpora" / SHADOW_RUN_ID
SHADOW_BINS_DIR = SHADOW_ROOT / "bins"
SHADOW_BINS_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_START_YEAR = 1986
BIN_YEARS = 5
END_YEAR = 2025
EARLIEST_BACKFILL_YEAR = 1975

MIN_EXAMPLES_PER_BIN = 500
MAX_EXAMPLES_PER_BIN = 800
MAX_EXAMPLES_PER_BIBCODE = 2
MAX_PAIR_OVERLAP_RATE = 0.80
MAX_BIBCODE_OVERLAP_RATE = 0.30
SEED = 42
rng = np.random.default_rng(SEED)

master_df = pd.read_parquet(MASTER_PARQUET)
required_cols = {
    "source_row_index", "mention_order", "year", "time_bin_current",
    "bibcode", "sent", "start", "end", "has_math",
}
missing = required_cols - set(master_df.columns)
if missing:
    raise ValueError(f"Master mention table missing required columns: {sorted(missing)}")

main_pairs_by_bin, main_bibcodes_by_bin, main_bin_counts = load_main_corpus_index(MAIN_CORPUS_DIR)
print("Loaded main corpus bins:", main_bin_counts)

old_bin_files = sorted(SHADOW_BINS_DIR.glob("*.jsonl"))
if old_bin_files:
    print(f"Removing {len(old_bin_files)} existing bin files from {SHADOW_BINS_DIR}")
    for p in old_bin_files:
        p.unlink()


## Write shadow bins
We introduce overlap-aware sampling, and generate an overlap report so that we can ensure that no data from *Analysis corpus* is contained in *Shadow corpus*.


---



In [ ]:
written_bins = []
final_bin_counts = {}
overlap_report = {}
skipped_overlap_bins = {}

fixed_bin_labels = sorted(master_df["time_bin_current"].dropna().astype(str).unique(), key=bin_start_year)

for bin_label in fixed_bin_labels:
    bin_end = int(bin_label.split("-")[1])

    if bin_label == f"{ANALYSIS_START_YEAR}-{ANALYSIS_START_YEAR + BIN_YEARS - 1}":
        resolved_start, examples = resolve_first_bin_from_master(
            master_df=master_df,
            rng=rng,
            min_examples=MIN_EXAMPLES_PER_BIN,
            first_bin_end=bin_end,
        )
        if resolved_start is None:
            print(f"{bin_label}: could not reach {MIN_EXAMPLES_PER_BIN}, skipping")
            continue
        bin_label_out = f"{resolved_start}-{bin_end}"
        examples = [{**ex, "time_bin": bin_label_out} for ex in examples]
    else:
        bin_label_out = bin_label
        subset = master_df[master_df["time_bin_current"] == bin_label].copy()
        subset = subset.sort_values(
            ["source_row_index", "mention_order", "bibcode", "start", "end"],
            kind="mergesort",
        )
        examples = materialize_rows(subset, time_bin_value=bin_label_out)
        examples = cap_per_bibcode(examples, rng)

    if len(examples) < MIN_EXAMPLES_PER_BIN:
        print(f"{bin_label_out}: only {len(examples)} examples after caps, skipping")
        continue

    target_total = min(MAX_EXAMPLES_PER_BIN, len(examples))
    selected, overlap_stats = select_shadow_examples(
        examples,
        bin_label_out=bin_label_out,
        target_total=target_total,
        rng=rng,
        main_pairs_by_bin=main_pairs_by_bin,
        main_bibcodes_by_bin=main_bibcodes_by_bin,
    )

    if len(selected) < MIN_EXAMPLES_PER_BIN:
        print(f"{bin_label_out}: only {len(selected)} shadow examples available after overlap-aware selection, skipping")
        continue

    overlap_report[bin_label_out] = overlap_stats
    if overlap_stats["shadow_pair_overlap_rate"] > MAX_PAIR_OVERLAP_RATE or overlap_stats["shadow_bibcode_overlap_rate"] > MAX_BIBCODE_OVERLAP_RATE:
        skipped_overlap_bins[bin_label_out] = {
            **overlap_stats,
            "skip_reason": "overlap_threshold",
            "max_pair_overlap_rate": MAX_PAIR_OVERLAP_RATE,
            "max_bibcode_overlap_rate": MAX_BIBCODE_OVERLAP_RATE,
        }
        print(
            f"{bin_label_out}: skipped due to overlap with main {overlap_stats['main_overlap_bin']} | "
            f"pair_overlap={overlap_stats['shadow_pair_overlap_rate']:.1%} "
            f"bibcode_overlap={overlap_stats['shadow_bibcode_overlap_rate']:.1%}"
        )
        continue

    selected = sorted(selected, key=lambda ex: (ex["year"], ex["bibcode"], ex["start"]))

    out_file = SHADOW_BINS_DIR / f"{bin_label_out}.jsonl"
    write_jsonl(selected, out_file)
    written_bins.append(bin_label_out)
    final_bin_counts[bin_label_out] = len(selected)

    yr_counts = pd.Series([ex["year"] for ex in selected]).value_counts().sort_index().to_dict()
    print(
        f"{bin_label_out}\n: wrote {len(selected)} examples -> {out_file}\n"
        f"pair_overlap={overlap_stats['shadow_pair_overlap_rate']:.1%} "
        f"bibcode_overlap={overlap_stats['shadow_bibcode_overlap_rate']:.1%} | "
        f"per-year: {yr_counts}"
    )

print("\nFinal shadow bin counts:")
print(final_bin_counts)



 ## Build and save the derived manifest for the annotation-support corpus 
---



In [ ]:
SHADOW_MANIFEST_PATH = SHADOW_ROOT / "corpus_manifest.json"

with open(MASTER_MANIFEST, "r", encoding="utf-8") as f:
    master_manifest = json.load(f)

prod_filter_keys = [
    "require_year",
    "require_abstract",
    "lowercase_abstract",
    "excluded_arxiv_categories",
    "excluded_doctypes",
    "biology_filter_terms",
]
derived_filters = {
    key: master_manifest["filters"][key]
    for key in prod_filter_keys
    if key in master_manifest.get("filters", {})
}

bin_content_hashes = {
    bin_label: file_sha256(SHADOW_BINS_DIR / f"{bin_label}.jsonl")
    for bin_label in written_bins
}

manifest = {
    "run_id": SHADOW_RUN_ID,
    "main_run_id": MAIN_RUN_ID,
    "source_parquet": master_manifest["source_parquet"],
    "output_dir": workspace_relative(SHADOW_BINS_DIR),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "start_year": ANALYSIS_START_YEAR,
    "end_year": END_YEAR,
    "bin_years": BIN_YEARS,
    "written_bins": written_bins,

    "filters": derived_filters,

    "sampling": {
        "min_examples_per_bin": MIN_EXAMPLES_PER_BIN,
        "max_examples_per_bin": MAX_EXAMPLES_PER_BIN,
        "max_examples_per_bibcode": MAX_EXAMPLES_PER_BIBCODE,
        "max_pair_overlap_rate": MAX_PAIR_OVERLAP_RATE,
        "max_bibcode_overlap_rate": MAX_BIBCODE_OVERLAP_RATE,
        "seed": SEED,
        "overlap_preference": [
            "fresh pair and fresh bibcode",
            "fresh pair with overlapping bibcode",
            "pair overlap only as last resort",
        ],
    },

    "main_bin_counts": main_bin_counts,
    "final_bin_counts": final_bin_counts,
    "final_total_examples": int(sum(final_bin_counts.values())),
    "overlap_report": overlap_report,
    "skipped_overlap_bins": skipped_overlap_bins,
    "bin_content_hashes": bin_content_hashes,
}

with open(SHADOW_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(f"Saved corpus manifest -> {SHADOW_MANIFEST_PATH}")



<br>

## Export annotation sets from the annotation-support corpus

---

<br>


In [ ]:
ANNOTATION_SET_ID = "shadow_balanced_1800"
ANNOTATION_SET_DIR = SHADOW_ROOT / "annotation-sets" / ANNOTATION_SET_ID
ANNOTATION_SET_DIR.mkdir(parents=True, exist_ok=True)

TARGET_PER_BIN = 300
MINI_PER_BIN = 100
SAMPLING_SEED = 42
SAMPLING_RNG = np.random.default_rng(SAMPLING_SEED)

FULL_SET_PATH = ANNOTATION_SET_DIR / f"{ANNOTATION_SET_ID}.jsonl"
FULL_SET_MANIFEST_PATH = ANNOTATION_SET_DIR / f"{ANNOTATION_SET_ID}_manifest.json"
MINI_SET_PATH = ANNOTATION_SET_DIR / f"{ANNOTATION_SET_ID}_mini_{MINI_PER_BIN}_per_bin.jsonl"
MINI_SET_MANIFEST_PATH = ANNOTATION_SET_DIR / f"{ANNOTATION_SET_ID}_mini_{MINI_PER_BIN}_per_bin_manifest.json"

POOL_QUOTAS = {
    "mixed": 40,
    "A": 70,
    "B": 70,
    "C": 70,
    "random": 50,
}

A_PAT = re.compile(
    r"\b("
    r"candidate|candidates|"
    r"composed of|consists of|constitute[s]?|"
    r"wimp|wimps|"
    r"axion|axions|"
    r"neutralino|neutralinos|"
    r"sterile neutrino|right-handed neutrino|"
    r"primordial black hole|primordial black holes|pbh|pbhs|"
    r"self-interacting|"
    r"annihilat(?:e|es|ed|ing|ion)?|"
    r"decay|decays|decaying|lifetime|"
    r"relic density|thermal relic|"
    r"cross section|cross-section|"
    r"wimp-nucleon|dark matter-nucleon|"
    r"millicharged|fuzzy dark matter|warm dark matter|hot dark matter|cold dark matter candidate|"
    r"ultralight|ultraheavy|"
    r"neutralino lsp|inert scalar|majorana fermion"
    r")\b",
    re.I,
)

B_PAT = re.compile(
    r"\b("
    r"rotation curve|rotation curves|"
    r"lensing|microlensing|"
    r"structure formation|"
    r"dark matter dominated|"
    r"dominated by dark matter|"
    r"halo|"
    r"density profile|"
    r"distribution of dark matter|"
    r"abundance|"
    r"content|"
    r"fraction|"
    r"required to|"
    r"explains|"
    r"accounts for|"
    r"constrain|"
    r"cluster|"
    r"galaxy|"
    r"void|"
    r"subhalo"
    r")\b",
    re.I,
)

C_PAT = re.compile(
    r"\b("
    r"cdm|hdm|wdm|chdm|lcdm|lambda cdm|cosmology|cosmological|model|models|scenario|framework|simulation|simulations|n-body|power spectrum|initial condition|nfw|einasto|press-schechter|density parameter|omega"
    r")\b",
    re.I,
)

with open(SHADOW_MANIFEST_PATH, "r", encoding="utf-8") as f:
    shadow_manifest = json.load(f)

annotation_rows = []
selected_by_bin = {}
bin_summaries = []

for bin_path in sorted(SHADOW_BINS_DIR.glob("*.jsonl"), key=lambda p: bin_start_year(p.stem)):
    rows = load_jsonl(bin_path)

    indexed_rows = []
    for idx, row in enumerate(rows):
        sentence = row["sent"]
        bucket, flags = classify_bucket(sentence)
        indexed_rows.append({
            "id": f"{bin_path.stem}_{idx}",
            "bin": bin_path.stem,
            "year": int(row["year"]),
            "bibcode": row.get("bibcode"),
            "sent": sentence,
            "sentence": sentence,
            "sentence_marked": highlight_span(sentence, row["start"], row["end"]),
            "start": int(row["start"]),
            "end": int(row["end"]),
            "source_idx": idx,
            "bucket": bucket,
            "bucket_flags": flags,
        })

    pools = {name: [] for name in ["mixed", "A", "B", "C", "other"]}
    for item in indexed_rows:
        pools[item["bucket"]].append(item)

    chosen_ids = set()
    bin_selected = []
    pool_taken = {name: 0 for name in pools}

    def take_from_pool(pool_name, n_take):
        chosen = []
        for item in choose_without_replacement(pools[pool_name], n_take):
            if item["id"] in chosen_ids:
                continue
            chosen.append(item)
            chosen_ids.add(item["id"])
        pool_taken[pool_name] += len(chosen)
        bin_selected.extend(chosen)

    take_from_pool("mixed", POOL_QUOTAS["mixed"])
    take_from_pool("A", POOL_QUOTAS["A"])
    take_from_pool("B", POOL_QUOTAS["B"])
    take_from_pool("C", POOL_QUOTAS["C"])

    remaining = [item for item in indexed_rows if item["id"] not in chosen_ids]
    random_take = choose_without_replacement(remaining, POOL_QUOTAS["random"])
    for item in random_take:
        chosen_ids.add(item["id"])
        bin_selected.append(item)
    pool_taken["other"] += sum(1 for item in random_take if item["bucket"] == "other")

    if len(bin_selected) < TARGET_PER_BIN:
        for item in remaining:
            if item["id"] in chosen_ids:
                continue
            chosen_ids.add(item["id"])
            bin_selected.append(item)
            if len(bin_selected) >= TARGET_PER_BIN:
                break

    bin_selected = sorted(bin_selected[:TARGET_PER_BIN], key=lambda item: item["source_idx"])
    selected_by_bin[bin_path.stem] = bin_selected
    annotation_rows.extend(bin_selected)

    bucket_counts = {
        name: sum(1 for item in indexed_rows if item["bucket"] == name)
        for name in pools
    }

    bin_summaries.append({
        "bin": bin_path.stem,
        "source_rows": len(indexed_rows),
        "selected_rows": len(bin_selected),
        "available_bucket_counts": bucket_counts,
        "selected_bucket_counts": dict(Counter(item["bucket"] for item in bin_selected)),
        "selected_year_counts": dict(Counter(item["year"] for item in bin_selected)),
        "pool_taken": dict(pool_taken),
    })

with open(FULL_SET_PATH, "w", encoding="utf-8") as f:
    for row in annotation_rows:
        payload = {k: v for k, v in row.items() if k not in {"bucket", "bucket_flags"}}
        f.write(json.dumps(payload, ensure_ascii=False) + "\n")

mini_rows = []
mini_bin_counts = {}

for bin_label, rows in selected_by_bin.items():
    chosen = choose_without_replacement(rows, min(MINI_PER_BIN, len(rows)))
    chosen = sorted(chosen, key=lambda item: item["source_idx"])
    mini_bin_counts[bin_label] = len(chosen)

    for row in chosen:
        mini_rows.append({
            "id": row["id"],
            "bin": row["bin"],
            "year": row["year"],
            "bibcode": row["bibcode"],
            "sentence": row["sentence"],
            "sentence_marked": row["sentence_marked"],
            "start": row["start"],
            "end": row["end"],
            "source_idx": row["source_idx"],
        })

with open(MINI_SET_PATH, "w", encoding="utf-8") as f:
    for row in mini_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved balanced annotation set ->", FULL_SET_PATH)
print("Saved mini annotation set ->", MINI_SET_PATH)
print("Full-set rows:", len(annotation_rows))
print("Mini-set rows:", len(mini_rows))
print("Counts by bin:", {k: len(v) for k, v in selected_by_bin.items()})


## Save derived manifests

---

In [ ]:

annotation_manifest = {
    "annotation_set_id": ANNOTATION_SET_ID,
    "shadow_run_id": SHADOW_RUN_ID,
    "shadow_manifest_path": workspace_relative(SHADOW_MANIFEST_PATH),
    "shadow_corpus_dir": workspace_relative(SHADOW_BINS_DIR),
    "source_parquet": shadow_manifest["source_parquet"],
    "created_at_utc": datetime.now(timezone.utc).isoformat(),

    "selection": {
        "target_per_bin": TARGET_PER_BIN,
        "mini_per_bin": MINI_PER_BIN,
        "seed": SAMPLING_SEED,
        "pool_quotas": POOL_QUOTAS,
        "bucket_logic": {
            "mixed": "two or more of A/B/C heuristic patterns fire",
            "A": "only A heuristic pattern fires",
            "B": "only B heuristic pattern fires",
            "C": "only C heuristic pattern fires",
            "other": "none of A/B/C heuristic patterns fire",
        },
    },

    "source_bins": shadow_manifest["written_bins"],
    "source_bin_counts": shadow_manifest["final_bin_counts"],
    "bin_summaries": bin_summaries,
    "total_rows": len(annotation_rows),
}

with open(FULL_SET_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(annotation_manifest, f, indent=2)

mini_manifest = {
    "annotation_set_id": ANNOTATION_SET_ID,
    "parent_set": FULL_SET_PATH.name,
    "parent_manifest": FULL_SET_MANIFEST_PATH.name,
    "shadow_run_id": SHADOW_RUN_ID,
    "shadow_manifest_path": workspace_relative(SHADOW_MANIFEST_PATH),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),

    "selection": {
        "mini_per_bin": MINI_PER_BIN,
        "seed": SAMPLING_SEED,
    },

    "bin_counts": mini_bin_counts,
    "total_rows": len(mini_rows),
}

with open(MINI_SET_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(mini_manifest, f, indent=2)

print("Saved balanced annotation manifest ->", FULL_SET_MANIFEST_PATH)
print("Saved mini annotation manifest ->", MINI_SET_MANIFEST_PATH)


## Shadow corpus created

The cells above built the shadow corpus from the validated master mention table. Below follows an optional step, where a full-scoped version of the Analysis corpus is created (for comparison, other analytical tasks, et.c).

---

---

## Lexical master

After we have obtained the shadow corpus, we can derive a final, more complete, lexical master corpus from the master corpus. This uncapped, larger, lexical corpus is derived directly from the master corpus after excluding the designated annotation set used for GPT seeding and SciBERT training/evaluation. The result is a richer, time-binned corpus namespace that can feed the standard embedding and reduction notebooks without leaking the held-out annotation set back upstream.



In [ ]:
LEXICAL_RUN_ID = "lexical_master"
LEXICAL_ROOT = LOCAL_DIR / "corpora" / LEXICAL_RUN_ID
LEXICAL_BINS_DIR = LEXICAL_ROOT / "bins"
LEXICAL_BINS_DIR.mkdir(parents=True, exist_ok=True)

with open(MASTER_MANIFEST, "r", encoding="utf-8") as f:
    master_manifest = json.load(f)

master_df = pd.read_parquet(MASTER_PARQUET)
required_cols = {
    "example_key",
    "source_row_index",
    "mention_order",
    "year",
    "time_bin_current",
    "bibcode",
    "sent",
    "start",
    "end",
    "has_math",
}
missing = required_cols - set(master_df.columns)
if missing:
    raise ValueError(f"Master mention table missing required columns: {sorted(missing)}")

annotation_rows = load_jsonl(FULL_SET_PATH)
annotation_example_keys = {example_key(row) for row in annotation_rows}
print(f"Annotation examples to exclude: {len(annotation_example_keys)}")

old_bin_files = sorted(LEXICAL_BINS_DIR.glob("*.jsonl"))
if old_bin_files:
    print(f"Removing {len(old_bin_files)} existing bin files from {LEXICAL_BINS_DIR}")
    for p in old_bin_files:
        p.unlink()

lexical_df = master_df[master_df["time_bin_current"].notna()].copy()
lexical_df = lexical_df[~lexical_df["example_key"].isin(annotation_example_keys)].copy()
lexical_df = lexical_df.sort_values(
    ["source_row_index", "mention_order", "bibcode", "start", "end"],
    kind="mergesort",
)

lexical_written_bins = []
lexical_final_bin_counts = {}
fixed_bin_labels = sorted(lexical_df["time_bin_current"].dropna().astype(str).unique(), key=bin_start_year)

for bin_label in fixed_bin_labels:
    subset = lexical_df[lexical_df["time_bin_current"].astype(str) == bin_label].copy()
    rows = materialize_lexical_rows(subset, time_bin_value=bin_label)
    if not rows:
        continue

    out_file = LEXICAL_BINS_DIR / f"{bin_label}.jsonl"
    write_jsonl(rows, out_file)
    lexical_written_bins.append(bin_label)
    lexical_final_bin_counts[bin_label] = len(rows)

    yr_counts = pd.Series([row["year"] for row in rows]).value_counts().sort_index().to_dict()
    print(f"{bin_label}: wrote {len(rows)} rows -> {out_file} | per-year: {yr_counts}")

print("\nFinal lexical bin counts:")
print(lexical_final_bin_counts)


## Build and save manifest

This manifest preserves master-corpus provenance, records the shadow corpus exclusion that protects downstream classifier work, and writes the fields expected by the embedding notebook:

- `written_bins`
- `n_written_bins`
- `final_bin_counts`
- `bin_content_hashes`

---

In [ ]:
LEXICAL_MANIFEST_PATH = LEXICAL_ROOT / "corpus_manifest.json"

prod_filter_keys = [
    "require_year",
    "require_abstract",
    "lowercase_abstract",
    "excluded_arxiv_categories",
    "excluded_doctypes",
    "biology_filter_terms",
]
derived_filters = {
    key: master_manifest["filters"][key]
    for key in prod_filter_keys
    if key in master_manifest.get("filters", {})
}

lexical_bin_content_hashes = {
    bin_label: file_sha256(LEXICAL_BINS_DIR / f"{bin_label}.jsonl")
    for bin_label in lexical_written_bins
}

manifest = build_derived_corpus_manifest(
    run_id=LEXICAL_RUN_ID,
    output_dir=LEXICAL_BINS_DIR,
    master_manifest=master_manifest,
    written_bins=lexical_written_bins,
    final_bin_counts=lexical_final_bin_counts,
    bin_content_hashes=lexical_bin_content_hashes,
    start_year=ANALYSIS_START_YEAR,
    end_year=END_YEAR,
    bin_years=BIN_YEARS,
    filters=derived_filters,
    sampling={
        "uncapped": True,
        "source_scope": "master_corpus rows with non-null time_bin_current",
        "exclude_annotation_set_id": ANNOTATION_SET_ID,
        "exclude_annotation_manifest": str(FULL_SET_MANIFEST_PATH),
        "excluded_annotation_rows": len(annotation_example_keys),
        "seed": None,
    },
    extra_fields={
        "annotation_set_id_excluded": ANNOTATION_SET_ID,
        "annotation_set_path_excluded": str(FULL_SET_PATH),
        "annotation_manifest_path_excluded": str(FULL_SET_MANIFEST_PATH),
        "source_rows_pre_exclusion": int(master_df[master_df["time_bin_current"].notna()].shape[0]),
        "excluded_example_keys": int(len(annotation_example_keys)),
        "rows_after_exclusion": int(sum(lexical_final_bin_counts.values())),
    },
)

with open(LEXICAL_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(f"Saved corpus manifest -> {LEXICAL_MANIFEST_PATH}")
